# non-Multilayers process

In [11]:
!pip install pennylane

In [12]:
import pennylane as qml
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

n_qubits = 6
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch")
def quantum_circuit(inputs, weights):
    for i in range(n_qubits):
        qml.RY(inputs[i], wires=i)

    qml.BasicEntanglerLayers(weights, wires=range(n_qubits))
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class QuanvolutionLayer(nn.Module):
    def __init__(self, n_qubits=4):
        super(QuanvolutionLayer, self).__init__()
        self.q_weights = nn.Parameter(0.01 * torch.rand((1, n_qubits)))
        self.wev_weights = nn.Parameter(torch.randn(n_qubits, 6))
        self.bias = nn.Parameter(torch.randn(6))
        self.n_qubits = n_qubits

    def forward(self, x):
        batch_size, height, width, channels = x.shape
        output = torch.zeros(batch_size, height - 1, width - 1, self.n_qubits, device=x.device)

        for i in range(height - 1):
            for j in range(width - 1):
                for b in range(batch_size):
                    patch = x[b, i:i+2, j:j+2, :].reshape(-1)
                    q_out = torch.tensor(quantum_circuit(patch, self.q_weights), dtype=torch.float32).detach().to(x.device)
                    output[b, i, j, :] = q_out

        output = torch.matmul(output, self.wev_weights) + self.bias
        return F.normalize(output, dim=-1)

class MultiTaskQCNN(nn.Module):
    def __init__(self, n_qubits=6):
        super(MultiTaskQCNN, self).__init__()
        self.quanv = QuanvolutionLayer(n_qubits)
        self.conv = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)  # Batch normalization
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.shared_fc = nn.Linear(16 * 3 * 3, 64)  # Shared FC layer

        # Separate heads for multitask outputs (6 heads for 6 outputs)
        self.head1 = nn.Linear(64, 1)
        self.head2 = nn.Linear(64, 1)
        self.head3 = nn.Linear(64, 1)
        self.head4 = nn.Linear(64, 1)
        self.head5 = nn.Linear(64, 1)
        self.head6 = nn.Linear(64, 1)

        self.dropout = nn.Dropout(0.3)
        nn.init.xavier_uniform_(self.shared_fc.weight)

    def forward(self, x):
        x = self.quanv(x)
        x = x.permute(0, 3, 1, 2)
        x = F.leaky_relu(self.bn1(self.conv(x)))
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dropout(F.leaky_relu(self.shared_fc(x)))

        # Separate heads for multitask predictions
        out1 = self.head1(x)
        out2 = self.head2(x)
        out3 = self.head3(x)
        out4 = self.head4(x)
        out5 = self.head5(x)
        out6 = self.head6(x)

        return torch.cat([out1, out2, out3, out4, out5, out6], dim=1).unsqueeze(-1)


In [14]:
data = np.load('inputs.npy')
labels = np.load('outputpH.npy')
print(data.shape, labels.shape)
labels = torch.tensor(labels, dtype=torch.float32)

(177, 7, 7, 6) (177, 6, 1)


In [15]:


model = MultiTaskQCNN()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

num_epochs = 40
for epoch in range(num_epochs):
    optimizer.zero_grad()
    outputs = model(data)
    print(type(outputs), type(labels))
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    print(f"epoch {epoch+1}/{num_epochs}\n loss: {loss.item():.4f}")


<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 1/40
 loss: 45.9056
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 2/40
 loss: 31.0933
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 3/40
 loss: 19.8708
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 4/40
 loss: 13.5478
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 5/40
 loss: 12.0959
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 6/40
 loss: 11.8288
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 7/40
 loss: 11.4930
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 8/40
 loss: 10.0692
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 9/40
 loss: 9.6410
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 10/40
 loss: 10.0331
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 11/40
 loss: 9.8634
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 12/40
 loss: 9.2951
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 13/40
 loss: 9.1207
<class 'torch.Tensor'> <class 'torch.Tensor'>
epoch 14/40
 loss:

In [16]:
torch.save(model.state_dict(), "quanvolutional_model.pth")